# Project Introduction
The project is to build a FAQ chatbot, that will help prospective students to get accurate answers about the LLM-Zoomcamp free course. The system will use as knowledge base the complete set of lessons hosted on Github repository online. Each lesson is a single markdown file.  

I will first create a "static RAG" system and then I will conver it into an "agentic RAG".
## Environment setup
For this project I will use the following setup:  
- `uv` a ultra-fast Python package manager
    - `uv init`
- The following python packages:
    - `gitsource` to retrieve the knowledge base from GitHub
    - `minsearch` to create an "in-memory" search engine for the knowledge base
    - `openai` to communicate with the LLM model
    - `jupyter` to work in a jupyter notebook
    - `python-dotenv` to load environmental variables
    - `toyaikit` to implement the agentic framework

## Phase 1: Static RAG

In the Phase 1, I will build the static version of this RAG System.
As the acronym suggest the main architecture will follow three major steps:

1. Retrieval of all course's lesson from the GitHub repository
2. Augmentation of the prompt with ***instructions***, ***context***, and the ***user query***
3. Generation of an output by the LLM to provide the user with an answer that is as accurate as possible  
    (we will see the limits of the static RAG mechanics)

### 1.1 Retrieval (fetch knowledge base)

In [1]:
# Retrieve all the course's lessons from the GitHub repository
from gitsource import GithubRepositoryDataReader

# Instatiate a GithubRepositoryDataReader object to read the course's lessons
repo_reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"}, # Use a set and not a list as it is collection of unique values and faster to lookup
    filename_filter=lambda path: "/lessons" in path,
)

# Read in the course's lessons
# <- list(RawRepositoryFile)
lesson_files = repo_reader.read()

In [2]:
# Find how many total lessons are in the LLM-Zoomcamp course
print(f"Found {len(lesson_files)} lessons")

Found 72 lessons


In [3]:
# Find what standard methods and attributes the RawRepositoryFile object has
methods = [method for method in dir(lesson_files[0]) if not method.startswith("_")]
print(methods)

['content', 'filename', 'parse']


In [4]:
# List to store each lesson file parsed into a dict
lesson_docs = []

# The method parse will parse each lesson RawRepositoryFile object into a dict
for lesson in lesson_files:
    lesson_docs.append(lesson.parse())

In [5]:
# Inspect first 3 parsed lessons
lesson_docs[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

### 1.2 Retrieval (indexing the knowledge base "`lesson_files`")
Indexing will make the entire knowledge searchable.
In this case I will use the library `minsearch` to implement the search functionality.

In [6]:
import minsearch

A `minsearch` implementation requires exactly three steps:

1. Creating a `minsearch` `Index` and declare as arguments what fields of our data we want to make ***searchable*** (`text_fields`) and which field will act as a filter to exclude entire documents from a search (`keyword_fields`).
2. Fitting the `Index`: We feed our full knowledge base to `minsearch` to create the *inverted index* to make our knwoledge base searchable and filterable.
3. Perform a search: We pass a question (`user_query`) to get back a ranked list of matching documents from our knowledge base.

In [7]:
# Print the keys of one obejct in our knowledge base to see what can be used as text_field and keyword_field in our minsearch implementation
list(lesson_docs[0])

['content', 'filename']

In [8]:
# Instatiate a minsearch Index object
lesson_index = minsearch.Index(
    text_fields=["content"],
    # This parameter is only needed if we use filter_dict in the search method
    keyword_fields=["filename"]
)

# Create the actual searchable index of the knowledge base (lesson_files)
lesson_index.fit(lesson_docs)

##### Making a first search in our knowledge base

In [9]:
user_query = "How does the agentic loop keep calling the model until it stops?"

# I only pass the user_query, as boost_dict is not needed as we only one text_fields
# Also, I won't pass the filter_dict parameter as we don't need to exclude any lesson from the search
# In this search we only keep the first result
lesson_index.search(user_query, num_results=1)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [10]:
search_results = lesson_index.search(user_query)

# Inspect first two results
search_results[:2]

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

### 1.3 Generation

I will use the Class RAGPipeline I defined in the `rag_schema` to make the entire process work and call the LLM to generate an answer to a user question.

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [12]:
# importlib.reload: re-runs rag_schema.py after editing it, without restarting the kernel.
# In notebooks, normal imports are cached, reload avoids cached code during development.
from importlib import reload
import rag_schema

reload(rag_schema)
from rag_schema import RAGPipeline

my_rag_pipeline = RAGPipeline(
    index_base=lesson_index,
    llm_client=OpenAI()
)

In [13]:
answer, tokens = my_rag_pipeline.execute_rag("How does the agentic loop keep calling the model until it stops?")

In [14]:
# IPython.display: renders LLM output as formatted Markdown in the notebook
# (plain `answer` shows a Python tuple with \n and no bold/lists).
from IPython.display import display, Markdown

display(Markdown(answer))
print(f"Input tokens: {tokens.input_tokens}")
print(f"Output tokens: {tokens.output_tokens}")
print(f"Total tokens: {tokens.total_tokens}")

The agentic loop uses a `while True` loop.

Each iteration it:
1. Sends the full message history to the model.
2. Checks the response for any `function_call` items.
3. If there are function calls, it runs the tool, appends the tool output to memory, and continues the loop.
4. If there are no function calls, it breaks and returns the final answer.

So the stop condition is simple: **if the model returns a response with no function calls, the loop ends**.

Input tokens: 7335
Output tokens: 111
Total tokens: 7446


### Chunking Implementation

Full lesson files are too long for precise retrieval, a keyword match in a 3000-word file makes the entire lesson file as context.
Chunking splits each lesson into smaller chunks (that slightly overlap at the boundaries) just before indexing the full knwoledge base.
Each chunk is like a sliding window of `size` characters that moves forward by `step` characters. Because `step < size`, 
consecutive chunks overlap, ensuring content at boundaries isn't lost.
Smaller chunks mean more precise search results and fewer input tokens sent to the LLM.

`Fecth Knowledge Base -> Chunking (parsed version) -> Search -> Augmentation -> Generation`

In [15]:
from gitsource import chunk_documents

# -> list(dict)
lesson_chunks = chunk_documents(lesson_docs, size=2000, step=1000)

print(f"The chunking process has created {len(lesson_chunks)} chunks")
# Inspect keys present in the "chunks" dictionary
print(lesson_chunks[0].keys())

display(Markdown(f"Starting at character #{str(lesson_chunks[1]['start'])}"))
display(Markdown(lesson_chunks[1]["content"]))
display(Markdown(lesson_chunks[1]["filename"]))

The chunking process has created 295 chunks
dict_keys(['start', 'content', 'filename'])


Starting at character #1000

the next
word based on what you typed so far.

A large language model does the same thing, but at a much larger scale.
It has billions of parameters and is trained on most of the text on the
internet. When it predicts the next word, it feels like you're talking
to an intelligent being. It understands what you ask and gives
meaningful answers.

In this course, we treat LLMs as black boxes. We won't look inside or
cover the theory, and we won't host a model ourselves. We use an LLM
provider and call it over an API. For us, an LLM is a box: text goes in,
text comes out.

But LLMs have limitations:

- Knowledge cutoff: they only know what was in their training data.
  If you ask about something that happened after training, they won't
  know - or worse, they'll make something up.
- No access to your data: they can't see your documents, databases,
  or internal systems unless you provide that information.
- Hallucinations: they sometimes produce confident-sounding answers
  that are simply wrong.

## The project

RAG solves these problems by giving the LLM relevant documents at
question time. We don't hope the model memorized the answer. We
retrieve the right information and hand it to the LLM, and the model
generates a grounded response. This lets us inject knowledge the model
never saw during training. That's why RAG is still the most common way
people use LLMs in the industry.

To make this concrete, we build a FAQ agent for our course. A student
asks something like "when does the course start?" and the agent answers
from the FAQ data we prepared.

This module has two parts.

In Part 1 (the next 9 lessons) we will:

- Understand what RAG is and how it works
- Build a search engine over a real FAQ dataset
- Write a prompt that combines the user's question with search results
- Wire it all together into a working RAG pipeline
- Split ingestion and query into separate processes

In Part 2, we make the pipeline agentic. The LLM decides when and
what to search, instead of 

01-agentic-rag/lessons/01-intro.md

# Implementing RAG with Chunking

In [16]:
# -> list(dict)
doc_chunks = chunk_documents(lesson_docs, size=2000, step=1000)

In [17]:
# Recreate a new minsearch Index object with the chunked knowledge base
# -> minsearch.Index
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
).fit(doc_chunks)

In [18]:
# Instatiate a new RAGPipeline object with the chunked knowledge base
my_pipeline_with_chunks = RAGPipeline(
    index_base=chunk_index,
    llm_client=OpenAI()
)

In [19]:
# -> tuple(str, dict)
answer, tokens = my_pipeline_with_chunks.execute_rag("How does the agentic loop keep calling the model until it stops?")

In [20]:
display(Markdown(answer))
print(f"Input tokens: {tokens.input_tokens}")
print(f"Output tokens: {tokens.output_tokens}")
print(f"Total tokens: {tokens.total_tokens}")

It uses a `while True` loop and a flag like `has_function_calls`.

Each iteration:
- calls the model
- checks the returned items
- if there is a `function_call`, it runs the tool, appends the result, and sets `has_function_calls = True`
- if there are no function calls in that turn, the loop breaks

So the loop stops when the model returns a final answer with no more tool calls.

Input tokens: 2516
Output tokens: 95
Total tokens: 2611


## Phase 2: Agentic RAG

In the Phase 2, I will build the agentic version of this RAG System.  

The main difference is that the agentic RAG will be able to call the LLM multiple times to generate an answer to a user question.

- The LLM decides when to call a tool and what arguments to pass
- My python code executes the actual function
- The result goes back to the LLM
- The LLM decides whether to call another tool or produce a final answer

This is what is called a "decision loop". Basically the decision of calling a tool lives with the LLM, but the execution still happens in my code.



### The ToyAIKit

ToyAIKit is a minimalistic Python library for building AI assistants powered by Large Language Models (LLMs).  

It provides a simple yet powerful framework for creating agentic conversational systems with advanced capabilities including:
- function calling
- tool integration
- multi-provider support.

### Why ToyAiKit?

So far `search` runs once, with the exact user query.  

To make this pipeline agentic I will have to provide the LLM with a tool, in this case to carry out a search in the knowledge base of the user query, and let the LLM decide what to search and when.

For learning purposes and simplicity I will use `toyaikit`, developed by [Alexey Grigorev](https://github.com/alexeygrigorev), a Python library that implements 
a minimal agentic framework, giving the LLM tools (functions), letting it decide when to call them, and looping until it produces a satisfactory final answer.  

`Toyaikit`does the same core thing as production frameworks like OpenAI Agents SDK, PydanticAI, or LangChain but without the extra complexity of those production frameworks, which ideal for early development phases or learnign purposes.

The module must be added to the project: `uv add toyaikit`

#### The tools
The first step is to create the "search" tool.  

This tool will be used to search the knowledge base for the user query.  


In [21]:
def search(query: str) -> list[dict]:
    """
    Search the knowledge base for the user query.
    """
    # Search in the knowledge base for the user query
    kb = chunk_index.search(query)
    return kb

#### Importing classes and functions I need from ToyAIKit

In [22]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

#### Adding the tools to ToyAIkit

When I defined the `search` function I did so by including a docstring and the type hint.  
ToyAIKit reads these automatically to build the tool schema (a json-like structured description the LLM needs to understand what the tool does and what inputs it expects).  

Without a type hint and docstring, the schema would be incomplete or missing, and the LLM wouldn't know how to use the tool correctly.

The list of the tools' schema can be see using the `Tools.get_tools() method`.

In [23]:
# Instantiating the Tools Class object
tools = Tools()

# Adding the search tool to the Tools object
tools.add_tool(search)

In [24]:
# Checking what toyaikit produced (the tool's schema to pass to the LLM)
tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the knowledge base for the user query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

#### The chat interface and agent



> **Note:**  
>`IPythonChatInterface` is optional and purely cosmetic.  
> It renders a chat box inside the notebook so whoever uses it can interact with the agent like a "mini ChatGPT".  
> Without it, they interact with the agent by calling `runner.run("your question")` 
> in a code cell and reading the plain text output.  
> Same agent, same results, different presentation.

In [25]:
chat_widget = IPythonChatInterface()
display_callback= DisplayingRunnerCallback(chat_widget)

In [26]:
# The instructions for the LLM
LLM_INSTRUCTIONS = f'''
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
'''.strip()

In [27]:
# Instantiating the OpenAIResponsesRunner object from the ToyAIKit used to interact with and OpenAI LLM
agent = OpenAIResponsesRunner(
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
    developer_prompt=LLM_INSTRUCTIONS,
    tools=tools,
    # The chat interface is used to display the agent's responses in the notebook (optional)
    chat_interface=chat_widget
)

#### Running one prompt

In [28]:
# start the agentic loop of the ToyAIKit Framework
user_query = "How does the agentic loop work, and how is it different from plain RAG?"

chat_widget.display("")

agent.loop(
    prompt=user_query,
    callback=display_callback
)


-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool.\nMake multiple searches with different keywords before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop plain RAG difference"}', call_id='call_DEHKp8MGf0KKNJDo9CdbaNhJ', name='search', type='function_call', id='fc_0b78f3d427444791006a3a743e35f8819f96daa3fa1339efbf', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop iterative retrieval tool use"}', call_id='call_BUSXFIEpPOTeb7THKdKdmIUg', name='search', type='function_call', id='fc_0b78f3d427444791006a3a743e3608819f88f371afa76a5fab', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"RAG versus agentic loop course notes"}', 